# AndinaLog 03B — Notebook 1: diagnóstico didáctico de Flota

**Objetivo:** revisar el maestro Bronze de camiones sin modificar sus cuatro columnas. Cada columna recibe `*_estado` y `*_motivo`; cada fila recibe un resumen y una decisión de cuarentena diagnóstica. No se usa catálogo ni motor externo. Se exportan únicamente **diagnosticado** y **cuarentena**.

## Criterios de negocio

- Centros permitidos: Cochabamba, La Paz, Santa Cruz, Oruro y Tarija, confirmados para este ejercicio.
- Tipos permitidos: `Seco` y `Refrigerado`, asumidos como catálogo cerrado del caso 03B.
- `capacidad_kg` se interpreta como capacidad de carga útil. Debe ser numérica y mayor que cero. Una capacidad menor que **750 kg** genera `REVISAR`, no cuarentena: es un umbral orientativo para esta flota, no un mínimo universal ni una restricción legal. El vehículo concreto tendría que verificarse con su ficha técnica.
- `camion_id` debe cumplir `CAM-##`. Copias exactas y claves con información contradictoria se diferencian. El diagnóstico registra, pero no normaliza ni elimina filas.

Los estados son `OK`, `REVISAR` y `CRITICO`. Una fila entra en cuarentena de diagnóstico si al menos una columna es `CRITICO`.

**Referencias para el criterio orientativo:** Mercedes-Benz publica una configuración refrigerada con 759 kg de carga útil (<https://conversion-world.mercedes-benz.com/en/GLOBAL/partner-produkt/3976/sprinter-coolkit-l3-h2-panel-van>); FUSO muestra que la carga útil cambia según modelo y configuración (<https://www.fuso-trucks.com/product/canter/7-5-tonnes/>). Ninguna fuente establece un mínimo general aplicable a todos los camiones.


In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-FLOTA-diagnostico-didactico-v1"
COLUMNAS_BRONZE = ["camion_id", "centro_distribucion_base", "capacidad_kg", "tipo_camion"]
CENTROS_PERMITIDOS = {"Cochabamba", "La Paz", "Santa Cruz", "Oruro", "Tarija"}
TIPOS_PERMITIDOS = {"Seco", "Refrigerado"}
UMBRAL_REVISION_CAPACIDAD_KG = 750

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "datasets/AndinaLog_03B_Bronce/andinalog_flota.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets/AndinaLog_03B_Bronce/andinalog_flota.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
RUTA_BRONZE = RAIZ / "datasets/AndinaLog_03B_Bronce/andinalog_flota.csv"
SALIDAS = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_flota/salidas"
HASH_BRONZE = hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
bronze = pd.read_csv(RUTA_BRONZE, dtype="string", encoding="utf-8-sig", keep_default_na=False)
if list(bronze.columns) != COLUMNAS_BRONZE:
    raise ValueError(f"Esquema Bronze inesperado: {list(bronze.columns)}")
df = bronze.copy(deep=True)
df.insert(0, "fila_bronze", range(1, len(df) + 1))
print("Filas Bronze:", len(df))


## Aplicación visible de reglas

`*_estado` indica la mayor severidad detectada para el campo y `*_motivo` conserva todas sus explicaciones. Las comprobaciones usan versiones auxiliares del texto, sin sobrescribir el valor Bronze.


In [ ]:
PRIORIDAD = {"OK": 0, "REVISAR": 1, "CRITICO": 2}
for columna in COLUMNAS_BRONZE:
    df[f"{columna}_estado"] = "OK"
    df[f"{columna}_motivo"] = ""

def marcar(columna, mascara, estado, motivo):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    e, m = f"{columna}_estado", f"{columna}_motivo"
    subir = mascara & df[e].map(PRIORIDAD).lt(PRIORIDAD[estado])
    df.loc[subir, e] = estado
    previo = df.loc[mascara, m]
    df.loc[mascara, m] = previo.where(previo.eq(""), previo + "; ") + motivo

for columna in COLUMNAS_BRONZE:
    marcar(columna, df[columna].str.strip().eq(""), "CRITICO", "Valor faltante")

# Identificador: formato y duplicidad exacta o contradictoria.
id_original = df["camion_id"]
id_limpio = id_original.str.strip()
id_normal = id_limpio.str.upper()
formato_id = id_original.str.fullmatch(r"CAM-\d{2}").fillna(False)
marcar("camion_id", id_limpio.ne("") & ~formato_id, "CRITICO", "Formato esperado: CAM-##")

firma = pd.util.hash_pandas_object(df[COLUMNAS_BRONZE], index=False)
variantes = firma.groupby(id_limpio, dropna=False).transform("nunique")
id_repetido = id_limpio.ne("") & id_limpio.duplicated(keep=False)
conflicto = id_repetido & variantes.gt(1)
copia = df.duplicated(COLUMNAS_BRONZE, keep="first") & ~conflicto
marcar("camion_id", conflicto, "CRITICO", "Mismo ID con datos contradictorios (revisar todas las variantes)")
marcar("camion_id", copia, "CRITICO", "Copia exacta posterior (no contar dos veces)")

# Una variante que colisiona con otro ID válido tras normalizar se revisa; no se fusiona.
colision_normal = (id_normal.ne("") & id_normal.duplicated(keep=False)
                  & ~id_limpio.duplicated(keep=False))
marcar("camion_id", colision_normal, "REVISAR", "Colisión potencial al normalizar espacios o mayúsculas")

centro = df["centro_distribucion_base"].str.strip()
marcar("centro_distribucion_base", centro.ne("") & ~centro.isin(CENTROS_PERMITIDOS),
       "CRITICO", "Centro fuera del catálogo permitido")

capacidad_texto = df["capacidad_kg"].str.strip()
capacidad = pd.to_numeric(capacidad_texto, errors="coerce")
marcar("capacidad_kg", capacidad_texto.ne("") & capacidad.isna(), "CRITICO", "Capacidad no numérica")
marcar("capacidad_kg", capacidad.notna() & capacidad.le(0), "CRITICO", "Capacidad debe ser mayor que 0 kg")
marcar("capacidad_kg", capacidad.gt(0) & capacidad.lt(UMBRAL_REVISION_CAPACIDAD_KG),
       "REVISAR", "Capacidad inferior a 750 kg: verificar ficha técnica del vehículo")

tipo = df["tipo_camion"].str.strip()
marcar("tipo_camion", tipo.ne("") & ~tipo.isin(TIPOS_PERMITIDOS),
       "CRITICO", "Tipo fuera del catálogo: Seco o Refrigerado")

print("Reglas aplicadas; columnas Bronze intactas")


## Resumen por fila y validación

La cuarentena del diagnóstico es un subconjunto del diagnosticado. No se debe concatenar con él al preparar tratamiento.


In [ ]:
estados = [f"{c}_estado" for c in COLUMNAS_BRONZE]
df["cantidad_columnas_con_problemas"] = df[estados].ne("OK").sum(axis=1)
df["en_cuarentena"] = df[estados].eq("CRITICO").any(axis=1)
df["severidad_maxima"] = "OK"
df.loc[df[estados].eq("REVISAR").any(axis=1), "severidad_maxima"] = "REVISAR"
df.loc[df[estados].eq("CRITICO").any(axis=1), "severidad_maxima"] = "CRITICO"

def resumen_fila(fila):
    columnas = [c for c in COLUMNAS_BRONZE if fila[f"{c}_estado"] != "OK"]
    textos = []
    for c in columnas:
        for motivo in fila[f"{c}_motivo"].split("; "):
            if motivo:
                texto = f"{c}: {motivo}"
                if texto not in textos: textos.append(texto)
    return pd.Series({"columnas_con_problemas": "|".join(columnas),
                      "motivos_fila": " | ".join(textos)})

df[["columnas_con_problemas", "motivos_fila"]] = df.apply(resumen_fila, axis=1)
df["version_diagnostico"] = VERSION_DIAGNOSTICO
df["sha256_bronze"] = HASH_BRONZE
cuarentena = df.loc[df["en_cuarentena"]].copy()

pd.testing.assert_frame_equal(df[COLUMNAS_BRONZE], bronze)
assert len(df) == len(bronze)
assert df["fila_bronze"].is_unique
assert cuarentena["motivos_fila"].ne("").all()
assert len(cuarentena) == int(df["en_cuarentena"].sum())
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest() == HASH_BRONZE
print("Diagnosticado:", len(df), "| Con problemas:", int(df["cantidad_columnas_con_problemas"].gt(0).sum()),
      "| Cuarentena:", len(cuarentena))
display(df[["fila_bronze", "camion_id", "camion_id_estado", "camion_id_motivo",
            "capacidad_kg_estado", "en_cuarentena", "motivos_fila"]].tail(15))


## Exportación

La salida diagnosticada contiene todas las filas y la cuarentena solo las filas críticas. El tratamiento posterior decidirá normalizaciones, exclusiones y cualquier revisión manual de conflictos.


In [ ]:
SALIDAS.mkdir(parents=True, exist_ok=True)
ruta_diagnosticado = SALIDAS / "andinalog_flota_didactico_v1_diagnosticado.csv"
ruta_cuarentena = SALIDAS / "andinalog_flota_didactico_v1_cuarentena.csv"
df.to_csv(ruta_diagnosticado, index=False, encoding="utf-8-sig")
cuarentena.to_csv(ruta_cuarentena, index=False, encoding="utf-8-sig")
print("Diagnosticado:", ruta_diagnosticado)
print("Cuarentena:", ruta_cuarentena)
